# Examen final: Sistema RAG sobre artículos de arXiv

Este cuaderno implementa:

1. Preparación completa del corpus.
2. Generación y reutilización de embeddings.
3. Almacenamiento persistente en ChromaDB.
4. Recuperación semántica.
5. Re-ranking con CrossEncoder.
6. Detección de consultas fuera del dominio.
7. Generación fundamentada con Gemini.
8. Presentación de evidencias.
9. Interfaz web con Gradio.

> El cuaderno reutiliza los archivos ya generados cuando existen, para evitar recalcular los embeddings o reconstruir ChromaDB innecesariamente.


## 0. Estructura esperada de la carpeta

```text
EXAMEN 2/
├── Notebook_RAG_arXiv_Gemini_LISTO.ipynb
├── .env
├── .gitignore
├── corpus/
│   ├── arxiv_data.csv
│   ├── arxiv_corpus_limpio.csv
│   ├── arxiv_embeddings.npy
│   └── arxiv_ids.npy
└── chroma_db/
```

El archivo `.env` debe contener:

```text
GEMINI_API_KEY=TU_CLAVE
```

El archivo `.gitignore` debe contener:

```text
.env
```


## 1. Instalación de dependencias

In [ ]:
%pip install -q pandas numpy sentence-transformers chromadb google-genai python-dotenv gradio

## 2. Importaciones y configuración general

In [1]:
import ast
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

RUTA_ENTRADA = Path("corpus/arxiv_data.csv")
RUTA_CORPUS_LIMPIO = Path("corpus/arxiv_corpus_limpio.csv")
RUTA_EMBEDDINGS = Path("corpus/arxiv_embeddings.npy")
RUTA_IDS = Path("corpus/arxiv_ids.npy")

RUTA_CHROMA = "chroma_db"
NOMBRE_COLECCION = "arxiv_papers"

MODELO_EMBEDDINGS = "sentence-transformers/all-MiniLM-L6-v2"
MODELO_RERANKER = "cross-encoder/ms-marco-MiniLM-L-6-v2"

MIN_LONGITUD_ABSTRACT = 100
BATCH_SIZE_EMBEDDINGS = 64
BATCH_SIZE_CHROMA = 5000

REGENERAR_CORPUS = False
REGENERAR_EMBEDDINGS = False
RECONSTRUIR_CHROMA = False

print("Configuración cargada.")


Configuración cargada.


## A. Preparación del corpus

La preparación normaliza las columnas `titles`, `summaries` y `terms`, elimina registros inválidos y duplicados, limpia las categorías y crea un documento único por paper.


In [2]:
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto)
    texto = texto.replace("\n", " ").replace("\r", " ").replace("\t", " ")
    texto = re.sub(r"\s+", " ", texto)
    return texto.strip()


def limpiar_categorias(valor):
    valor = limpiar_texto(valor)

    if not valor:
        return ""

    try:
        categorias = ast.literal_eval(valor)

        if isinstance(categorias, (list, tuple, set)):
            return ", ".join(
                str(categoria).strip()
                for categoria in categorias
                if str(categoria).strip()
            )
    except (ValueError, SyntaxError):
        pass

    return valor


def construir_documento(fila):
    partes = [f"Title: {fila['title']}"]

    if fila["categories"]:
        partes.append(f"Categories: {fila['categories']}")

    partes.append(f"Abstract: {fila['abstract']}")
    return "\n".join(partes)


In [3]:
if RUTA_CORPUS_LIMPIO.exists() and not REGENERAR_CORPUS:
    df_corpus = pd.read_csv(RUTA_CORPUS_LIMPIO)
    print("Se reutilizó el corpus limpio guardado.")
else:
    if not RUTA_ENTRADA.exists():
        raise FileNotFoundError(
            f"No se encontró el archivo original: {RUTA_ENTRADA.resolve()}"
        )

    df_original = pd.read_csv(RUTA_ENTRADA)

    columnas_esperadas = {"titles", "summaries", "terms"}
    faltantes = columnas_esperadas.difference(df_original.columns)

    if faltantes:
        raise ValueError(f"Faltan columnas obligatorias: {sorted(faltantes)}")

    df = df_original[["titles", "summaries", "terms"]].copy()

    df.rename(
        columns={
            "titles": "title",
            "summaries": "abstract",
            "terms": "categories"
        },
        inplace=True
    )

    df["title"] = df["title"].map(limpiar_texto)
    df["abstract"] = df["abstract"].map(limpiar_texto)
    df["categories"] = df["categories"].map(limpiar_categorias)

    cantidad_inicial = len(df)

    df = df[
        df["title"].ne("") &
        df["abstract"].ne("")
    ].copy()

    df["abstract_length"] = df["abstract"].str.len()
    df = df[df["abstract_length"] >= MIN_LONGITUD_ABSTRACT].copy()

    df.drop_duplicates(
        subset=["title", "abstract"],
        keep="first",
        inplace=True
    )

    df.reset_index(drop=True, inplace=True)

    df["paper_id"] = [
        f"arxiv_{indice:07d}"
        for indice in range(len(df))
    ]

    df["document"] = df.apply(construir_documento, axis=1)
    df["document_length"] = df["document"].str.len()
    df["word_count"] = df["document"].str.split().str.len()

    df_corpus = df.reset_index(drop=True).copy()

    if df_corpus["paper_id"].duplicated().any():
        raise ValueError("Se generaron identificadores duplicados.")

    if df_corpus["document"].duplicated().any():
        raise ValueError("El corpus todavía contiene documentos duplicados.")

    RUTA_CORPUS_LIMPIO.parent.mkdir(parents=True, exist_ok=True)
    df_corpus.to_csv(RUTA_CORPUS_LIMPIO, index=False, encoding="utf-8")

    print("Registros iniciales:", cantidad_inicial)
    print("Registros finales:", len(df_corpus))
    print("Corpus limpio guardado en:", RUTA_CORPUS_LIMPIO.resolve())

print("Dimensiones del corpus:", df_corpus.shape)
df_corpus.head()


Se reutilizó el corpus limpio guardado.
Dimensiones del corpus: (38983, 8)


,title,abstract,categories,abstract_length,paper_id,document,document_length,word_count
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"cs.CV, cs.LG",847,arxiv_0000000,Title: Survey on Semantic Stereo Matching / Se...,952,146
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"cs.CV, cs.AI, cs.LG",1500,arxiv_0000001,Title: FUTURE-AI: Guiding Principles and Conse...,1675,226
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","cs.CV, cs.AI",1645,arxiv_0000002,Title: Enforcing Mutual Consistency of Hard Re...,1779,243
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,cs.CV,1458,arxiv_0000003,Title: Parameter Decoupling Strategy for Semi-...,1571,211
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","cs.CV, cs.LG",1820,arxiv_0000004,Title: Background-Foreground Segmentation for ...,1941,287


## B. Representación mediante embeddings

Se utiliza `all-MiniLM-L6-v2`, que genera vectores de 384 dimensiones. Los vectores se normalizan para búsqueda con distancia coseno.


In [4]:
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer(MODELO_EMBEDDINGS)

ids_documentos = df_corpus["paper_id"].astype(str).to_numpy()

if RUTA_EMBEDDINGS.exists() and RUTA_IDS.exists() and not REGENERAR_EMBEDDINGS:
    embeddings = np.load(RUTA_EMBEDDINGS)
    ids_guardados = np.load(RUTA_IDS, allow_pickle=True)

    archivos_validos = (
        embeddings.ndim == 2
        and embeddings.shape[0] == len(df_corpus)
        and embeddings.shape[1] == 384
        and len(ids_guardados) == len(df_corpus)
        and np.array_equal(ids_guardados.astype(str), ids_documentos)
    )

    if not archivos_validos:
        raise ValueError(
            "Los embeddings guardados no coinciden con el corpus. "
            "Establece REGENERAR_EMBEDDINGS = True."
        )

    print("Se reutilizaron los embeddings guardados.")
else:
    textos = df_corpus["document"].astype(str).tolist()

    embeddings = modelo_embeddings.encode(
        textos,
        batch_size=BATCH_SIZE_EMBEDDINGS,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    if embeddings.shape != (len(df_corpus), 384):
        raise ValueError(f"Dimensiones inesperadas: {embeddings.shape}")

    np.save(RUTA_EMBEDDINGS, embeddings)
    np.save(RUTA_IDS, ids_documentos)

    print("Embeddings generados y guardados.")

print("Embeddings:", embeddings.shape)
print("Tipo:", embeddings.dtype)


Se reutilizaron los embeddings guardados.
Embeddings: (38983, 384)
Tipo: float32


## C. Almacenamiento en ChromaDB

La colección persiste en la carpeta `chroma_db`. Si ya contiene todos los documentos, se reutiliza.


In [5]:
import chromadb

cliente_chroma = chromadb.PersistentClient(path=RUTA_CHROMA)

colecciones_existentes = {
    item.name
    for item in cliente_chroma.list_collections()
}

if NOMBRE_COLECCION in colecciones_existentes and RECONSTRUIR_CHROMA:
    cliente_chroma.delete_collection(name=NOMBRE_COLECCION)
    colecciones_existentes.remove(NOMBRE_COLECCION)

if NOMBRE_COLECCION in colecciones_existentes:
    coleccion = cliente_chroma.get_collection(name=NOMBRE_COLECCION)
    print("Se reutilizó la colección existente.")
else:
    coleccion = cliente_chroma.create_collection(
        name=NOMBRE_COLECCION,
        metadata={"hnsw:space": "cosine"}
    )
    print("Colección creada.")


Se reutilizó la colección existente.


In [6]:
if coleccion.count() == 0:
    total = len(df_corpus)

    for inicio in range(0, total, BATCH_SIZE_CHROMA):
        fin = min(inicio + BATCH_SIZE_CHROMA, total)
        lote = df_corpus.iloc[inicio:fin]

        metadatos = [
            {
                "title": str(fila.title),
                "categories": str(fila.categories),
                "abstract": str(fila.abstract)
            }
            for fila in lote.itertuples(index=False)
        ]

        coleccion.add(
            ids=lote["paper_id"].astype(str).tolist(),
            documents=lote["document"].astype(str).tolist(),
            embeddings=embeddings[inicio:fin].tolist(),
            metadatas=metadatos
        )

        print(f"Insertados: {fin}/{total}")
else:
    print("La colección ya contiene datos; no se insertaron nuevamente.")

if coleccion.count() != len(df_corpus):
    raise ValueError(
        f"ChromaDB contiene {coleccion.count()} documentos, "
        f"pero el corpus tiene {len(df_corpus)}."
    )

print("Total en ChromaDB:", coleccion.count())


La colección ya contiene datos; no se insertaron nuevamente.
Total en ChromaDB: 38983


## D. Recuperación semántica

In [7]:
def buscar_documentos(consulta, top_k=15):
    if not isinstance(consulta, str):
        raise TypeError("La consulta debe ser texto.")

    consulta = consulta.strip()

    if not consulta:
        raise ValueError("La consulta no puede estar vacía.")

    if not isinstance(top_k, int) or top_k <= 0:
        raise ValueError("top_k debe ser un entero mayor que cero.")

    embedding_consulta = modelo_embeddings.encode(
        consulta,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    resultados = coleccion.query(
        query_embeddings=[embedding_consulta.tolist()],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    documentos = []

    for posicion, paper_id in enumerate(resultados["ids"][0]):
        metadata = resultados["metadatas"][0][posicion] or {}
        distancia = float(resultados["distances"][0][posicion])

        documentos.append({
            "rank": posicion + 1,
            "paper_id": paper_id,
            "title": metadata.get("title", ""),
            "categories": metadata.get("categories", ""),
            "abstract": metadata.get("abstract", ""),
            "document": resultados["documents"][0][posicion],
            "distance": distancia,
            "similarity": 1.0 - distancia
        })

    return documentos


## E. Re-ranking

In [8]:
from sentence_transformers import CrossEncoder

modelo_reranker = CrossEncoder(MODELO_RERANKER)


def rerank_documentos(consulta, documentos_recuperados, top_n=5):
    if not documentos_recuperados:
        return []

    if not isinstance(top_n, int) or top_n <= 0:
        raise ValueError("top_n debe ser un entero mayor que cero.")

    pares = [
        [consulta, documento["document"]]
        for documento in documentos_recuperados
    ]

    scores = modelo_reranker.predict(
        pares,
        show_progress_bar=False
    )

    documentos_reordenados = []

    for documento, score in zip(documentos_recuperados, scores):
        copia = documento.copy()
        copia["rerank_score"] = float(score)
        documentos_reordenados.append(copia)

    documentos_reordenados.sort(
        key=lambda item: item["rerank_score"],
        reverse=True
    )

    documentos_finales = documentos_reordenados[:top_n]

    for posicion, documento in enumerate(documentos_finales, start=1):
        documento["rerank_rank"] = posicion

    return documentos_finales


## F. Conexión segura con Gemini

In [9]:
from dotenv import load_dotenv
from google import genai

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError(
        "No se encontró GEMINI_API_KEY. "
        "Crea un archivo .env en la carpeta del proyecto."
    )

cliente_gemini = genai.Client(api_key=api_key)

print("Cliente de Gemini creado correctamente.")


Cliente de Gemini creado correctamente.


## G. Generación aumentada por recuperación (RAG)

In [ ]:
def construir_contexto(documentos, max_caracteres=5000):
    if not documentos:
        return ""

    bloques = []
    total = 0

    for indice, documento in enumerate(documentos, start=1):
        bloque = (
            f"[Document {indice}]\n"
            f"Title: {documento.get('title', '')}\n"
            f"Categories: {documento.get('categories', '')}\n"
            f"Abstract: {documento.get('abstract', '')}\n"
        )

        if total + len(bloque) > max_caracteres:
            break

        bloques.append(bloque)
        total += len(bloque)

    return "\n".join(bloques)


def generar_respuesta(prompt):
    respuesta = cliente_gemini.models.generate_content(
        model="gemini-3.5-flash",
        contents=prompt
    )

    texto = getattr(respuesta, "text", None)

    if not texto or not texto.strip():
        return (
            "The corpus does not contain enough information "
            "to answer this question."
        )

    return texto.strip()


In [11]:
RESPUESTA_FUERA_DE_DOMINIO = (
    "Lo siento, esta consulta no está relacionada con el dominio "
    "del corpus de artículos científicos de arXiv o no existe "
    "información suficiente para responderla."
)


def responder_rag(
    consulta,
    top_k_recuperacion=15,
    top_n_reranking=5,
    umbral_similitud=0.30,
    umbral_reranking=-2.0
):
    if not isinstance(consulta, str):
        raise TypeError("La consulta debe ser texto.")

    consulta = consulta.strip()

    if not consulta:
        raise ValueError("La consulta no puede estar vacía.")

    candidatos = buscar_documentos(
        consulta,
        top_k=top_k_recuperacion
    )

    if not candidatos:
        return {
            "consulta": consulta,
            "respuesta": RESPUESTA_FUERA_DE_DOMINIO,
            "evidencias": [],
            "fuera_de_dominio": True
        }

    documentos_finales = rerank_documentos(
        consulta,
        candidatos,
        top_n=top_n_reranking
    )

    if not documentos_finales:
        return {
            "consulta": consulta,
            "respuesta": RESPUESTA_FUERA_DE_DOMINIO,
            "evidencias": [],
            "fuera_de_dominio": True
        }

    mejor_documento = documentos_finales[0]
    mejor_similitud = float(mejor_documento.get("similarity", 0.0))
    mejor_rerank = float(
        mejor_documento.get("rerank_score", float("-inf"))
    )

    fuera_de_dominio = (
        mejor_similitud < umbral_similitud
        or mejor_rerank < umbral_reranking
    )

    if fuera_de_dominio:
        return {
            "consulta": consulta,
            "respuesta": RESPUESTA_FUERA_DE_DOMINIO,
            "evidencias": documentos_finales,
            "fuera_de_dominio": True,
            "mejor_similitud": mejor_similitud,
            "mejor_rerank_score": mejor_rerank
        }

    contexto = construir_contexto(
        documentos_finales,
        max_caracteres=5000
    )

    if not contexto.strip():
        return {
            "consulta": consulta,
            "respuesta": RESPUESTA_FUERA_DE_DOMINIO,
            "evidencias": documentos_finales,
            "fuera_de_dominio": True,
            "mejor_similitud": mejor_similitud,
            "mejor_rerank_score": mejor_rerank
        }

    prompt = f"""
You are an academic assistant specialized in scientific papers.

Answer the question using ONLY the retrieved context.

Instructions:
- Answer directly and clearly.
- Write between 3 and 5 complete sentences.
- Focus only on what the question asks.
- Combine information from multiple documents when possible.
- Do not invent information.
- Do not use external knowledge.
- Cite the supporting evidence using [Document 1], [Document 2], etc.
- If the context is insufficient, answer exactly:
  "The corpus does not contain enough information to answer this question."

Context:
{contexto}

Question:
{consulta}

Final answer:
"""

    respuesta = generar_respuesta(prompt)

    return {
        "consulta": consulta,
        "respuesta": respuesta,
        "evidencias": documentos_finales,
        "contexto": contexto,
        "fuera_de_dominio": False,
        "mejor_similitud": mejor_similitud,
        "mejor_rerank_score": mejor_rerank
    }


## H. Presentación de evidencias

In [12]:
def mostrar_resultado_rag(resultado):
    print("=" * 90)
    print("CONSULTA")
    print("=" * 90)
    print(resultado.get("consulta", ""))

    print("\n" + "=" * 90)
    print("RESPUESTA")
    print("=" * 90)
    print(resultado.get("respuesta", ""))

    print("\n" + "=" * 90)
    print("EVIDENCIAS")
    print("=" * 90)

    if resultado.get("fuera_de_dominio", False):
        print("Consulta identificada como fuera del dominio.")

        if "mejor_similitud" in resultado:
            print(
                "Mejor similitud:",
                round(resultado["mejor_similitud"], 4)
            )

        if "mejor_rerank_score" in resultado:
            print(
                "Mejor score de re-ranking:",
                round(resultado["mejor_rerank_score"], 4)
            )

        return

    for indice, evidencia in enumerate(
        resultado.get("evidencias", []),
        start=1
    ):
        print("\n" + "-" * 90)
        print(f"DOCUMENTO {indice}")
        print("Paper ID:", evidencia.get("paper_id", ""))
        print("Título:", evidencia.get("title", ""))
        print("Categorías:", evidencia.get("categories", ""))
        print(
            "Similitud:",
            round(evidencia.get("similarity", 0.0), 4)
        )
        print(
            "Score de re-ranking:",
            round(evidencia.get("rerank_score", 0.0), 4)
        )
        print("\nAbstract:")
        print(evidencia.get("abstract", "")[:800])


## I. Pruebas

In [13]:
resultado = responder_rag(
    "What are the main applications of Graph Neural Networks?",
    top_k_recuperacion=15,
    top_n_reranking=3
)

mostrar_resultado_rag(resultado)


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use a newer model for the latest features and improvements.', 'status': 'NOT_FOUND'}}

In [ ]:
resultado_fuera = responder_rag(
    "How can I prepare an Ecuadorian encebollado?"
)

mostrar_resultado_rag(resultado_fuera)


## J. Interfaz web con Gradio

In [ ]:
def formatear_evidencias(evidencias):
    if not evidencias:
        return "No se encontraron evidencias."

    bloques = []

    for indice, evidencia in enumerate(evidencias, start=1):
        bloque = f"""
### Documento {indice}

**Título:** {evidencia.get("title", "No disponible")}

**Paper ID:** `{evidencia.get("paper_id", "No disponible")}`

**Categorías:** {evidencia.get("categories", "No disponible")}

**Similitud:** `{evidencia.get("similarity", 0.0):.4f}`

**Score de re-ranking:** `{evidencia.get("rerank_score", 0.0):.4f}`

**Abstract:**

{evidencia.get("abstract", "")[:800]}
"""
        bloques.append(bloque)

    return "\n\n---\n\n".join(bloques)


def chat(consulta):
    if not isinstance(consulta, str) or not consulta.strip():
        return (
            "Por favor, ingrese una consulta válida.",
            "No hay evidencias disponibles."
        )

    try:
        resultado = responder_rag(consulta)

        if resultado.get("fuera_de_dominio", False):
            evidencias = (
                "⚠️ La consulta fue identificada como fuera "
                "del dominio del corpus científico."
            )
        else:
            evidencias = formatear_evidencias(
                resultado.get("evidencias", [])
            )

        return resultado["respuesta"], evidencias

    except Exception as error:
        return (
            f"No fue posible procesar la consulta: {error}",
            "No se pudieron mostrar evidencias."
        )


In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(
        label="Consulta",
        lines=3,
        placeholder=(
            "Example: What are the main applications "
            "of Graph Neural Networks?"
        )
    ),
    outputs=[
        gr.Textbox(
            label="Respuesta generada",
            lines=8
        ),
        gr.Markdown()
    ],
    title="Sistema RAG sobre artículos científicos de arXiv",
    description=(
        "Recuperación semántica, re-ranking, detección fuera "
        "del dominio y generación fundamentada mediante Gemini."
    ),
    examples=[
        ["What are the main applications of Graph Neural Networks?"],
        ["How is reinforcement learning used in robotics?"],
        ["Recent advances in diffusion models for image generation."]
    ]
)

demo.launch()


## K. Despliegue

Para desplegar en Hugging Face Spaces:

1. Exportar las funciones y la interfaz a `app.py`.
2. Subir `chroma_db/`.
3. Crear `requirements.txt`.
4. En **Settings → Variables and secrets**, crear el secreto `GEMINI_API_KEY`.
5. No subir el archivo `.env`.
